# ***LakeShore240***

This contains a minimal experiment using the
[`lakeshore`](https://pypi.org/project/lakeshore/) library to interact with
Lake Shore Model 240 temperature monitors.

___

## **Installation**

### *Window*
1. Plug the Lake Shore 240 USB module into your PC.  
2. Windows will automatically detect the device and list it under **Device Manager → Ports (COM & LPT)**.  
   No additional driver installation is required.

### *Linux or MacOS*
For Linux, you need to install [`cp210x`](https://www.silabs.com/developer-tools/usb-to-uart-bridge-vcp-drivers) driver manually.
Or, Alternative
```bash
sudo nano /etc/udev/rules.d/99-lakeshore240.rules
```
then, put this script on the file
```bash
ACTION=="add", \
  ATTRS{idVendor}=="1fb9", ATTRS{idProduct}=="0205", \
  RUN+="/sbin/modprobe cp210x", \
  RUN+="/bin/sh -c 'echo 1fb9 0205 > /sys/bus/usb-serial/drivers/cp210x/new_id'"
```

For checking that device is connected.
```bash
sudo dmesg | tail 20
```
it should appear
```bash
cp210x converter now attached to ttyUSB0
```
Then you are ready for communicate to LakeShore240 

___

## **LakeShore Driver**

First, you need to install lakeshore driver
```bash
pip install lakeshore
```

Then you can import the library


[`documentation`](https://lake-shore-python-driver.readthedocs.io/en/latest/model_240.html)

In [128]:
%pip install lakeshore

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [129]:
from lakeshore import Model240
from lakeshore.model_240 import Model240InputParameter, Model240CurveHeader

### **Connecting device**

For connecting to module

In [130]:
module = Model240()

Be sure that therea are no other source connecting to LakeShore240 then there will be error occur

### **LakeShore Setting**

### *input parameter*

For the input parameter which is determine the behavior of the input

| Field | Type | Description |
|-------|------|-------------|
| `sensor` | `Model240SensorTypes` | Sensor type connected to the input. Must be a member of the `Model240SensorTypes` enum. |
| `auto_range_enable` | `bool` | **True** → automatic range selection <br> **False** → manual range (`input_range` is used). |
| `current_reversal_enable` | `bool` | Enable current-reversal (cancels thermal-EMF errors on resistive sensors). Always **False** for diode channels. |
| `units` | `Model240Units` | Preferred display/measurement units. Member of the `Model240Units` enum. |
| `input_enable` | `bool` | **True** → channel enabled <br> **False** → channel disabled. |
| `input_range` | `Model240InputRange` | Fixed measurement range used **only** when `auto_range_enable` is **False**. Member of the `Model240InputRange` enum. |

#### `InputRange` — manual ranges **and their factory-set excitation currents**

| Enum member | Code | Resistance span | Default excitation |
|-------------|:----:|-----------------|--------------------|
| `RANGE_NTCRTD_10_OHMS`  | **0** | 0 – 10 Ω  | **1 mA** |
| `RANGE_NTCRTD_30_OHMS`  | **1** | 0 – 30 Ω  | **300 µA** |
| `RANGE_NTCRTD_100_OHMS` | **2** | 0 – 100 Ω | **100 µA** |
| `RANGE_NTCRTD_300_OHMS` | **3** | 0 – 300 Ω | **30 µA** |
| `RANGE_NTCRTD_1_KIL_OHMS` | **4** | 0 – 1 kΩ  | **10 µA** |
| `RANGE_NTCRTD_3_KIL_OHMS` | **5** | 0 – 3 kΩ  | **3 µA** |
| `RANGE_NTCRTD_10_KIL_OHMS`| **6** | 0 – 10 kΩ | **1 µA** |
| `RANGE_NTCRTD_30_KIL_OHMS`| **7** | 0 – 30 kΩ | **300 nA** |
| `RANGE_NTCRTD_100_KIL_OHMS`| **8** | 0 – 100 kΩ | **100 nA** |
| `RANGE_DIODE` (mV) | **0** | 0 – 3 mV | Voltage-mode; current is set internally by the instrument |
| `RANGE_PTRTD_1_KIL_OHMS` | **0** | 0 – 1 kΩ | Uses platinum RTD excitation (1 mA max), but this preset is seldom required |


Use these values **only** when `auto_range_enable` is **False**:


First, let get the input paramter for channal 1

In [131]:
in_param = module.get_input_parameter(channel=1)

In [132]:
in_param.__dict__

{'sensor_type': <SensorTypes.NTC_RTD: 3>,
 'temperature_unit': <Units.SENSOR: 3>,
 'auto_range_enable': True,
 'current_reversal_enable': True,
 'input_enable': True,
 'input_range': 0}

Setting the configuration to specific channel

In [133]:
param = Model240InputParameter(sensor=Model240.SensorTypes.NTC_RTD,
                               auto_range_enable=False,
                               current_reversal_enable=True,
                               units=Model240.Units.SENSOR,
                               input_enable=True,
                               input_range=Model240.InputRange.RANGE_NTCRTD_100_KIL_OHMS)

In [134]:
module.set_input_parameter(3, param)

Checking the result

In [135]:
module.get_input_parameter(channel=3).__dict__

{'sensor_type': <SensorTypes.NTC_RTD: 3>,
 'temperature_unit': <Units.SENSOR: 3>,
 'auto_range_enable': True,
 'current_reversal_enable': True,
 'input_enable': True,
 'input_range': 8}

### *Curve Header and Curve data points*

#### Curve-header fields

| Field | Type | Description |
|-------|------|-------------|
| `curve_name` | `str` | Human-readable name that appears in the front-panel/GUI curve list. |
| `serial_number` | `str` | Sensor serial, batch code, or any short ID you find useful. |
| `curve_data_format` | `Model240CurveFormat` | Units used for the *dY/dT* data points:<br>• `VOLTS_PER_KELVIN`  (2) for diodes<br>• `OHMS_PER_KELVIN`   (3) for linear-scale RTDs<br>• `LOG_OHMS_PER_KELVIN` (4) for log-scale RTDs |
| `temperature_limit` | `float` | Upper-temperature validity limit of the curve. (The instrument assumes the lower limit is 0 K.) |
| `coefficient` | `Model240TemperatureCoefficient` | Sign of the sensor’s temperature coefficient:<br>`POSITIVE` (2) or `NEGATIVE` (1). |

> **Note That**
> The coefficient will determine from the data point only, so you need to sort the data points for correct behavior (NTC or PTC).

First let get the curve header from specific channel

In [136]:
curve_head = module.get_curve_header(curve=1)

In [137]:
curve_head.__dict__

{'curve_name': 'User Curve 1   ',
 'serial_number': '          ',
 'curve_data_format': <CurveFormat.VOLTS_PER_KELVIN: 2>,
 'temperature_limit': 375.0,
 'coefficient': <TemperatureCoefficient.NEGATIVE: 1>}

For configuration curve header

In [138]:
header = Model240CurveHeader(curve_name="Channel5",
                             serial_number="676767",
                             curve_data_format=Model240.CurveFormat.OHMS_PER_KELVIN,
                             temperature_limit=400,
                             coefficient=Model240.Coefficients.NEGATIVE)

In [139]:
module.set_curve_header(input_channel=5, curve_header=header)

Check the result

In [140]:
module.get_curve_header(5).__dict__

{'curve_name': 'Channel5       ',
 'serial_number': '676767    ',
 'curve_data_format': <CurveFormat.OHMS_PER_KELVIN: 3>,
 'temperature_limit': 400.0,
 'coefficient': <TemperatureCoefficient.NEGATIVE: 1>}

#### Curve-Data-Points

| Field | Type | Valid values / format | Description |
|-------|------|-----------------------|-------------|
| `index` | `int` | **1 – 200** | Position of the point in the user-curve table. A curve can store up to 200 points and the instrument expects them in ascending order. |
| `units_value` | `float` | `±nnnnnn` (six significant digits) | Sensor reading corresponding to this point—Volts for diodes, Ohms for RTDs, or log-Ohms if the curve format is logarithmic. |
| `temp_value` | `float` | `+nnnnnn` (six significant digits, Kelvin) | Temperature that pairs with `units_value`. |


**Tips**

* All numbers must contain **exactly six digits** (no scientific notation).  
* Define the curve header first before loading points.  
* Leave `auto_range_enable=True` while collecting calibration data, then switch to manual if desired.

Let get data points from channel 1

In [147]:
for i in range(1, 201):
    print(module.get_curve_data_point(5, i))

0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0.000000
0.000000,0

For the first column is sensor unit and second column is temperature.

The Curve header coefficient field will auto determine with the `arrange of data points` in temperature column, So be careful. 

**Configuration curve data**

For this alternative will use .csv with pandas to export the data to Module

In [142]:
import pandas as pd

In [143]:
df = pd.read_csv("temperature_resistance_calibration.csv")
df.head()

,Temperature_K,Resistance_Ohms
0,233.15,277200.0
1,234.15,263600.0
2,235.15,250100.0
3,236.15,236800.0
4,237.15,224000.0


> For good practical, I recommended to set all data on the channel to be 0 before putting your data into it

In [146]:
import time
from time import sleep
for i in range(200):
    module.set_curve_data_point(channel=5, index=i+1, units=0, temp=0)
    sleep(0.1)

In [ ]:
for i, (temp, res) in enumerate(zip(df['Temperature_K'][::-1], df['Resistance_Ohms'][::-1])):
    print(f"index : {i+1}, sensor : {res}, temp : {temp}")
    module.set_curve_data_point(channel=5, index=i+1, units=res, temp=temp)
    sleep(0.1)

index : 1, sensor : 144.1, temp : 432.15
index : 2, sensor : 147.4, temp : 431.15
index : 3, sensor : 150.7, temp : 430.15
index : 4, sensor : 154.1, temp : 429.15
index : 5, sensor : 157.6, temp : 428.15
index : 6, sensor : 161.2, temp : 427.15
index : 7, sensor : 165.0, temp : 426.15
index : 8, sensor : 168.8, temp : 425.15
index : 9, sensor : 172.8, temp : 424.15
index : 10, sensor : 176.9, temp : 423.15
index : 11, sensor : 181.1, temp : 422.15
index : 12, sensor : 185.5, temp : 421.15
index : 13, sensor : 190.0, temp : 420.15
index : 14, sensor : 194.6, temp : 419.15
index : 15, sensor : 199.4, temp : 418.15
index : 16, sensor : 204.4, temp : 417.15
index : 17, sensor : 209.5, temp : 416.15
index : 18, sensor : 214.8, temp : 415.15
index : 19, sensor : 220.2, temp : 414.15
index : 20, sensor : 225.8, temp : 413.15
index : 21, sensor : 231.6, temp : 412.15
index : 22, sensor : 237.5, temp : 411.15
index : 23, sensor : 243.7, temp : 410.15
index : 24, sensor : 250.0, temp : 409.15
i

___

### **Monitoring data from LakeShore240**

There are many method for this module. For the full version please visit here [`documentation`](https://lake-shore-python-driver.readthedocs.io/en/latest/model_240.html)

In [145]:
module.get_identification()

InstrumentException: Communication timed out

In [ ]:
module.get_modname()

'Bashame Meme'

In [ ]:
module.get_sensor_name(1)

'RTD'

In [ ]:
module.get_channel_reading_status(1)

{'invalid reading': False,
 '': False,
 'temp under range': False,
 'temp over range': False,
 'sensor units over range': False,
 'sensor units under range': True}

In [ ]:
module.get_sensor_reading(1)

0.0

In [ ]:
module.get_sensor_units_channel_reading(1)

'+000.000'

In [ ]:
module.get_celsius_reading(5)

'+02.3129'

In [ ]:
module.get_kelvin_reading(5)

275.46

In [ ]:
module.get_filter(1)

'100'

In [ ]:
# module.disconnect_usb()